# C00 - Base canonica de preprocesamiento

Este notebook construye la base comun para todos los experimentos nuevos de preprocesamiento. La idea es separar con claridad que columnas son datos de entrada y cuales son auxiliares del challenge, convertir el marcador artificial `-999.0` a `NaN`, mapear el target a una convencion binaria y crear un split `train/valid` reproducible.

Este paso no aprende estadisticas del dataset completo: no imputa, no escala, no codifica categoricas, no selecciona features y no crea variables nuevas. Su funcion es dejar una representacion canonica y auditable para que los pipelines `P00` a `P04` se ajusten solo con `X_train_raw`.

Salidas principales en `data/preprocesor/C00_common/`:
- `X_train_raw.csv`, `X_valid_raw.csv`, `X_test_raw.csv` con `-999 -> NaN`.
- `y_train.csv`, `y_valid.csv`, pesos e identificadores.
- `metadata.json`, `split_summary.csv` y conteos de faltantes.



## Imports y configuracion


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_work_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        direct = candidate / "preprocessing_lib.py"
        nested = candidate / "50. preprocesamiento" / "preprocessing_lib.py"
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / "50. preprocesamiento"
    raise FileNotFoundError("No se pudo ubicar 50. preprocesamiento/preprocessing_lib.py")


work_dir = find_work_dir()
if str(work_dir) not in sys.path:
    sys.path.insert(0, str(work_dir))

from preprocessing_lib import (
    build_common_data,
    output_root,
    save_common_artifacts,
    save_global_pipeline_artifacts,
    save_p03_artifacts,
    split_summary,
    transform_global_pipeline,
    transform_p03,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)



## Construccion de la base comun


In [2]:
common = build_common_data()
output_dir = output_root() / "C00_common"

print(f"Directorio de salida: {output_dir}")
print("Resumen del split:")
display(split_summary(common).round(6))

missing_df = pd.DataFrame({
    "feature": list(common.metadata["missing_counts_by_column"].keys()),
    "missing_count": list(common.metadata["missing_counts_by_column"].values()),
}).sort_values("missing_count", ascending=False)
print("Columnas con mas faltantes luego de -999 -> NaN:")
display(missing_df.head(15))

print("Distribucion de PRI_jet_num en training completo:")
print(common.metadata["pri_jet_num_distribution_train_full"])


Directorio de salida: /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common
Resumen del split:


,split,rows,class_b,class_s,weight_sum
0,train,225000,147900.0,77100.0,370822.739836
1,valid,25000,16433.0,8567.0,40869.096094
2,test,550000,NaN,NaN,NaN


Columnas con mas faltantes luego de -999 -> NaN:


,feature,missing_count
4,DER_deltaeta_jet_jet,177457
5,DER_mass_jet_jet,177457
12,DER_lep_eta_centrality,177457
6,DER_prodeta_jet_jet,177457
26,PRI_jet_subleading_pt,177457
28,PRI_jet_subleading_phi,177457
27,PRI_jet_subleading_eta,177457
23,PRI_jet_leading_pt,99913
24,PRI_jet_leading_eta,99913
25,PRI_jet_leading_phi,99913


Distribucion de PRI_jet_num en training completo:
{0: 99913, 1: 77544, 2: 50379, 3: 22164}


## Guardado de artefactos


In [3]:
saved_paths = save_common_artifacts(common, output_dir)
print(f"Artefactos guardados: {len(saved_paths)}")
for path in saved_paths:
    print(f"- {path}")


Artefactos guardados: 13
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/X_train_raw.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/X_valid_raw.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/X_test_raw.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/y_train.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/y_valid.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/train_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/valid_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/train_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/valid_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/test_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/split_summary.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/C00_common/missing_counts_b